In [ ]:
from google.colab import drive
try:
  import awkward as ak
except:
  !pip install awkward
  import awkward as ak
import os
import numpy as np

drive.mount('/content/drive')

#200k data used for inference, maximum possible with google colab free tier
voxel_path3 = "/content/drive/MyDrive/dataset_voxel3.npy"
voxel_path5 = "/content/drive/MyDrive/dataset_voxel5.npy"
target_path3 = "/content/drive/MyDrive/target_energy3.npy"
target_path5 = "/content/drive/MyDrive/target_energy5.npy"
parquet_path3 = "/content/drive/MyDrive/parquet_chunks/chunk_300000.parquet"
parquet_path5 = "/content/drive/MyDrive/parquet_chunks/chunk_500000.parquet"
model_path = "/content/drive/MyDrive/cnn3d_final_model9.pth"
results_path = "/content/drive/MyDrive/Results_energy_regression"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
dataset3 = np.load(voxel_path3)
dataset5 = np.load(voxel_path5)
target_energy3 = np.load(target_path3)
target_energy5 = np.load(target_path5)
energies_ak3 = ak.from_parquet(parquet_path3, columns=["energy"])
energies_ak5 = ak.from_parquet(parquet_path5, columns=["energy"])
energies3 = ak.to_numpy(ak.sum(energies_ak3["energy"], axis=1))
energies5 = ak.to_numpy(ak.sum(energies_ak5["energy"], axis=1))
dataset = np.concatenate((dataset3, dataset5), axis=0)
target_energy = np.concatenate((target_energy3, target_energy5), axis = 0)
np.log1p(dataset*0.01, out=dataset)
np.log1p(target_energy, out=target_energy)

array([4.944873 , 4.840551 , 3.3308618, ..., 4.500124 , 5.5065336,
       5.2901535], dtype=float32)

In [ ]:
energies = np.concatenate((energies3, energies5), axis = 0)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

class HGCALDataset(Dataset):
    def __init__(self, voxels, targets):
        self.x = torch.from_numpy(voxels).float()
        self.y = torch.from_numpy(targets).float()

        self.x = self.x.unsqueeze(1)

        if self.y.ndim == 1:
            self.y = self.y.unsqueeze(1)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

hgc_data = HGCALDataset(dataset, target_energy)

In [ ]:
class HGCAL_Net(nn.Module):
    def __init__(self, depth=28, max_energy_log=5.8):
        super().__init__()
        self.depth = depth
        self.max_val = max_energy_log


        self.conv1 = nn.Conv3d(1, 32, kernel_size=(1, 3, 3), padding=(0, 1, 1))
        self.conv2 = nn.Conv3d(32, 64, kernel_size=(3, 1, 1), padding=(1, 0, 0))
        self.bn2 = nn.BatchNorm3d(64)

        self.conv3 = nn.Conv3d(64, 128, kernel_size=3, padding=(1, 2, 2), dilation=(1, 2, 2))
        self.bn3 = nn.BatchNorm3d(128)
        self.bottleneck = nn.Conv3d(128, 32, kernel_size=1)

        self.spatial_pool = nn.AdaptiveAvgPool3d((2, 4, 4))
        self.global_pool = nn.AdaptiveAvgPool3d(1)

        input_fc_dim = (32 * 2 * 4 * 4) + 32 + 1 + 1 + self.depth + 1

        self.fc = nn.Sequential(
            nn.Linear(input_fc_dim, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.SiLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        x_lin = torch.expm1(x)
        g_sum = torch.sum(x_lin.view(x_lin.size(0), -1), dim=1, keepdim=True)
        g_sum_log_norm = torch.log1p(g_sum) / self.max_val

        E_z = torch.sum(x_lin, dim=(1, 3, 4))
        E_z_norm = torch.log1p(E_z) / self.max_val

        z_indices = torch.arange(self.depth, device=x.device, dtype=x.dtype)
        z_cg = torch.sum(E_z * z_indices, dim=1, keepdim=True) / (g_sum + 1e-6)
        z_cg_norm = z_cg / self.depth
        
        lat_spread = torch.std(x_lin, dim=(2, 3, 4)).mean(dim=1, keepdim=True)
        lat_spread_norm = torch.tanh(lat_spread)

        x_c = F.silu(self.conv1(x))
        x_c = F.silu(self.bn2(self.conv2(x_c)))
        x_c = F.silu(self.bn3(self.conv3(x_c)))
        x_c = self.bottleneck(x_c)

        x_spat = self.spatial_pool(x_c).view(x_c.size(0), -1)
        x_glob = self.global_pool(x_c).view(x_c.size(0), -1)

        combined = torch.cat([
            x_spat, x_glob,
            g_sum_log_norm, z_cg_norm, E_z_norm, lat_spread_norm
        ], dim=1)

        return self.fc(combined)

In [ ]:
test_loader  = DataLoader(hgc_data,  batch_size=128, shuffle=False, num_workers= 2)

In [ ]:
model = HGCAL_Net()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        all_preds.append(outputs.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

y_pred_log = np.concatenate(all_preds).flatten()
y_true_log = np.concatenate(all_targets).flatten()
y_pred_gev = np.expm1(y_pred_log)
y_true_gev = np.expm1(y_true_log)

#saving data into numpy arrays for later use in a local environment for plotting and analysis
np.save("/content/drive/MyDrive/Results_energy_regression/y_pred_gev2", y_pred_gev)
np.save("/content/drive/MyDrive/Results_energy_regression/y_true_gev2", y_true_gev)
np.save("/content/drive/MyDrive/Results_energy_regression/energies_reco2", energies)